Отлично! Давай сделаем сразу оба варианта (1️⃣ упрощённый с OHE для CatBoost и 3️⃣ продвинутый с cat_features) на твоих данных train и test.

Вариант 1️⃣ — Все OHE (для LGBM, XGB, CatBoost)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# ==============================
# Подготовка данных
# ==============================
target_col = 'SalePrice'
X = train.drop(columns=[target_col, 'Id'])
y = train[target_col]


# Логарифмируем таргет
y_log = np.log1p(y)

# Разделяем train/validation
X_train, X_val, y_train_log, y_val_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

# Категориальные признаки
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Для LGBM и XGB нужно закодировать категориальные признаки числами
X_train_enc = X_train.copy()
X_val_enc = X_val.copy()
for col in cat_cols:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train_enc[col].astype(str))
    X_val_enc[col] = le.transform(X_val_enc[col].astype(str))

# ==============================
# Инициализация моделей
# ==============================
lgbm_model = LGBMRegressor(n_estimators=5000, learning_rate=0.05, random_state=42)
xgb_model = XGBRegressor(n_estimators=5000, learning_rate=0.05, random_state=42)
cat_model = CatBoostRegressor(n_estimators=5000, learning_rate=0.05, verbose=0, random_state=42)

# ==============================
# Обучение моделей с early stopping
# ==============================
lgbm_model.fit(
    X_train_enc, y_train_log,
    eval_set=[(X_val_enc, y_val_log)],
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbose=50
)

xgb_model.fit(
    X_train_enc, y_train_log,
    eval_set=[(X_val_enc, y_val_log)],
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbose=50
)

cat_model.fit(X_train, y_train_log, cat_features=cat_cols)

# ==============================
# Предсказание и ансамблирование
# ==============================
pred_lgbm = np.expm1(lgbm_model.predict(X_val_enc))
pred_xgb = np.expm1(xgb_model.predict(X_val_enc))
pred_cat = np.expm1(cat_model.predict(X_val))

# Soft voting — усреднение предсказаний
pred_ensemble = (pred_lgbm + pred_xgb + pred_cat) / 3

# ==============================
# Оценка
# ==============================
rmse = np.sqrt(mean_squared_error(np.expm1(y_val_log), pred_ensemble))
print(f"Ensemble RMSE: {rmse:.2f}")


ValueError: y contains previously unseen labels: 'Membran'

Вариант 2️⃣ — CatBoost с cat_features + LGBM/XGB с OHE

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# ==============================
# Подготовка данных
# ==============================
target_col = 'SalePrice'
X = train.drop(columns=[target_col, 'Id'])  # Id не нужен
y = train[target_col]
# y = np.log1p(train[target_col])  # логарифмируем таргет

# Разделяем train/validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Определяем категориальные признаки
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# ==============================
# Label Encoding с защитой от новых категорий
# ==============================
X_train_enc = X_train.copy()
X_val_enc = X_val.copy()

for col in cat_cols:
    le = LabelEncoder()
    
    # Заполняем NaN строкой 'missing'
    X_train_enc[col] = X_train_enc[col].astype(str).fillna('missing')
    X_val_enc[col] = X_val_enc[col].astype(str).fillna('missing')
    
    # Fit LabelEncoder на train
    le.fit(X_train_enc[col])
    X_train_enc[col] = le.transform(X_train_enc[col])
    
    # Преобразуем val: новые категории -> 'missing'
    X_val_enc[col] = X_val_enc[col].apply(lambda x: x if x in le.classes_ else 'missing')
    # Добавляем 'missing' в классы LabelEncoder, если её ещё нет
    if 'missing' not in le.classes_:
        le.classes_ = np.append(le.classes_, 'missing')
    X_val_enc[col] = le.transform(X_val_enc[col])

# ==============================
# Инициализация моделей
# ==============================
lgbm_model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42)
xgb_model = XGBRegressor(n_estimators=1000, learning_rate=0.05, random_state=42)
cat_model = CatBoostRegressor(n_estimators=1000, learning_rate=0.05, verbose=0, random_state=42)

# ==============================
# Обучение моделей
# ==============================
lgbm_model.fit(X_train_enc, y_train)
xgb_model.fit(X_train_enc, y_train)

# Для CatBoost заполняем NaN строкой 'missing' прямо в train
X_train_cat = X_train.copy()
X_val_cat = X_val.copy()
for col in cat_cols:
    X_train_cat[col] = X_train_cat[col].astype(str).fillna('missing')
    X_val_cat[col] = X_val_cat[col].astype(str).fillna('missing')

cat_model.fit(X_train_cat, y_train, cat_features=cat_cols)

# ==============================
# Предсказание и ансамблирование
# ==============================
pred_lgbm = lgbm_model.predict(X_val_enc)
pred_xgb = xgb_model.predict(X_val_enc)
pred_cat = cat_model.predict(X_val_cat)

# Soft voting — усреднение предсказаний
pred_ensemble = (pred_lgbm + pred_xgb + pred_cat) / 3

# ==============================
# Оценка
# ==============================
rmse = np.sqrt(mean_squared_error(y_val, pred_ensemble))
print(f"Ensemble RMSE: {rmse:.2f}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000827 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3129
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 73
[LightGBM] [Info] Start training from score 181441.541952
Ensemble RMSE: 26039.81


In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# ==============================
# Подготовка данных
# ==============================
target_col = 'SalePrice'
X = train.drop(columns=[target_col, 'Id'])  # Id не нужен
y = np.log1p(train[target_col])  # логарифмируем таргет

# Разделяем train/validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Определяем категориальные признаки
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# ==============================
# Label Encoding с защитой от новых категорий
# ==============================
X_train_enc = X_train.copy()
X_val_enc = X_val.copy()

for col in cat_cols:
    le = LabelEncoder()
    
    # Заполняем NaN строкой 'missing'
    X_train_enc[col] = X_train_enc[col].astype(str).fillna('missing')
    X_val_enc[col] = X_val_enc[col].astype(str).fillna('missing')
    
    # Fit LabelEncoder на train
    le.fit(X_train_enc[col])
    X_train_enc[col] = le.transform(X_train_enc[col])
    
    # Преобразуем val: новые категории -> 'missing'
    X_val_enc[col] = X_val_enc[col].apply(lambda x: x if x in le.classes_ else 'missing')
    # Добавляем 'missing' в классы LabelEncoder, если её ещё нет
    if 'missing' not in le.classes_:
        le.classes_ = np.append(le.classes_, 'missing')
    X_val_enc[col] = le.transform(X_val_enc[col])

# ==============================
# Инициализация моделей
# ==============================
lgbm_model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42)
xgb_model = XGBRegressor(n_estimators=1000, learning_rate=0.05, random_state=42)
cat_model = CatBoostRegressor(n_estimators=1000, learning_rate=0.05, verbose=0, random_state=42)

# ==============================
# Обучение моделей
# ==============================
lgbm_model.fit(X_train_enc, y_train)
xgb_model.fit(X_train_enc, y_train)

# Для CatBoost заполняем NaN строкой 'missing' прямо в train
X_train_cat = X_train.copy()
X_val_cat = X_val.copy()
for col in cat_cols:
    X_train_cat[col] = X_train_cat[col].astype(str).fillna('missing')
    X_val_cat[col] = X_val_cat[col].astype(str).fillna('missing')

cat_model.fit(X_train_cat, y_train, cat_features=cat_cols)

# ==============================
# Предсказание и обратное преобразование
# ==============================
pred_lgbm = np.expm1(lgbm_model.predict(X_val_enc))
pred_xgb = np.expm1(xgb_model.predict(X_val_enc))
pred_cat = np.expm1(cat_model.predict(X_val_cat))

# Soft voting — усреднение предсказаний
pred_ensemble = (pred_lgbm + pred_xgb + pred_cat) / 3

# ==============================
# Оценка RMSE
# ==============================
rmse = np.sqrt(mean_squared_error(np.expm1(y_val), pred_ensemble))
print(f"Ensemble RMSE: {rmse:.2f}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000994 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3129
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 73
[LightGBM] [Info] Start training from score 12.030658
Ensemble RMSE: 27057.75


In [ ]:
import pandas as pd
import numpy as np

# Обратное преобразование, если использовался log1p
pred_lgbm_orig = np.expm1(pred_lgbm)
pred_xgb_orig  = np.expm1(pred_xgb)
pred_cat_orig  = np.expm1(pred_cat)
pred_ensemble_orig = np.expm1(pred_ensemble)

# Собираем в DataFrame
val_results = X_val.copy()
val_results['y_true'] = y_val
val_results['pred_lgbm'] = pred_lgbm_orig
val_results['pred_xgb'] = pred_xgb_orig
val_results['pred_cat'] = pred_cat_orig
val_results['pred_ensemble'] = pred_ensemble_orig

# Добавим столбец ошибки ансамбля
val_results['error_ensemble'] = np.abs(val_results['y_true'] - val_results['pred_ensemble'])

# Сортируем по ошибке по убыванию
val_results_sorted = val_results.sort_values(by='error_ensemble', ascending=False)

# Выводим топ-10 примеров с наибольшей ошибкой
print(val_results_sorted.head(10))


In [ ]:
# Собираем DataFrame с предсказаниями
val_results = X_val.copy()
val_results['y_true'] = y_val
val_results['pred_lgbm'] = pred_lgbm_orig
val_results['pred_xgb']  = pred_xgb_orig
val_results['pred_cat']  = pred_cat_orig
val_results['pred_ensemble'] = pred_ensemble_orig

# Рассчитываем ошибки в процентах
val_results['error_lgbm_%'] = 100 * (val_results['pred_lgbm'] - val_results['y_true']) / val_results['y_true']
val_results['error_xgb_%']  = 100 * (val_results['pred_xgb'] - val_results['y_true']) / val_results['y_true']
val_results['error_cat_%']  = 100 * (val_results['pred_cat'] - val_results['y_true']) / val_results['y_true']
val_results['error_ensemble_%'] = 100 * (val_results['pred_ensemble'] - val_results['y_true']) / val_results['y_true']

# Сортируем по абсолютной ошибке ансамбля по убыванию
val_results_sorted = val_results.reindex(
    val_results['error_ensemble_%'].abs().sort_values(ascending=False).index
)

# Выводим топ-10 самых больших ошибок ансамбля
print(val_results_sorted.head(10)[[
    'y_true', 'pred_lgbm', 'pred_xgb', 'pred_cat', 'pred_ensemble',
    'error_lgbm_%', 'error_xgb_%', 'error_cat_%', 'error_ensemble_%'
]])
